# Classical baselines and feature diagnostics

Run this notebook cell by cell. It is intended to answer:

1. How strong are simple tabular baselines before ESN/QRC?
2. Are compact vs expanded volatility features redundant?
3. Which features appear to matter?
4. Does PCA/correlation pruning preserve most of the signal?
5. What should we try next?

In [ ]:
from pathlib import Path
import os

# Make notebook robust when launched from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

print(Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.baselines.esn import COMPACT_FEATURES, EXPANDED_FEATURES, TARGET
from qpitome_qrc.baselines.classical import (
    default_model_configs,
    fit_classical_baseline,
    permutation_importance_table,
    run_classical_suite,
)
from qpitome_qrc.data.loaders import load_market_stress_data
from qpitome_qrc.data.splits import (
    chronological_tabular_split,
    describe_splits,
    split_arrays,
)
from qpitome_qrc.dimred.feature_select import (
    CorrelationPruner,
    correlation_summary,
)
from qpitome_qrc.dimred.reducers import (
    DimReductionConfig,
    build_dimred_pipeline,
    pca_explained_variance_table,
)
from qpitome_qrc.evaluation.plots import (
    ensure_dir,
    save_correlation_heatmap,
    save_feature_importance_plot,
    save_metric_barplot,
    save_pca_variance_plot,
    save_pr_roc_curves,
)
from qpitome_qrc.evaluation.reports import build_pdf_report

## 1. Load processed dataset

This uses `data/processed/market_stress_v0.csv`.

If this fails, rerun:

```bash
python scripts/prepare_dataset.py
```

In [ ]:
df = load_market_stress_data()
print(df.shape)
df.head()

In [ ]:
compact = [f for f in COMPACT_FEATURES if f in df.columns]
expanded = [f for f in EXPANDED_FEATURES if f in df.columns]

print("compact features:", len(compact))
print(compact)
print("\nexpanded features:", len(expanded))
print(expanded)

## 2. Chronological split

Same split logic as before:

- train: before 2016
- validation: 2016–2019
- test: 2020 onward

In [ ]:
splits = chronological_tabular_split(df)
split_summary = describe_splits(splits, TARGET)
split_summary

## 3. Feature redundancy: correlations

This helps test whether our engineered volatility features are largely redundant.

In [ ]:
corr_compact = correlation_summary(df, compact)
corr_compact.head(20)

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[compact].corr().abs()
plt.imshow(corr, vmin=0, vmax=1, aspect="auto")
plt.colorbar(label="absolute correlation")
plt.xticks(range(len(compact)), compact, rotation=90)
plt.yticks(range(len(compact)), compact)
plt.title("Compact feature absolute correlation")
plt.tight_layout()
plt.show()

In [ ]:
if len(expanded) > len(compact):
    corr_expanded = correlation_summary(df, expanded)
    display(corr_expanded.head(30))

## 4. PCA diagnostic

PCA is fit on the train split only. This is not yet a model; it checks how many independent directions the compact feature set has.

In [ ]:
pca_config = DimReductionConfig(kind="pca", n_components=len(compact), scale=True)
pca_pipe = build_dimred_pipeline(pca_config)

X_train_compact = splits["train"][compact].to_numpy(dtype=float)
pca_pipe.fit(X_train_compact)

pca_summary = pca_explained_variance_table(pca_pipe)
pca_summary

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(
    pca_summary["component"],
    pca_summary["cumulative_explained_variance"],
    marker="o",
)
plt.xlabel("PCA component")
plt.ylabel("Cumulative explained variance")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.title("PCA cumulative explained variance")
plt.tight_layout()
plt.show()

## 5. Build feature sets for baseline comparison

We compare:

- compact features
- expanded features, if available
- compact correlation-pruned features
- compact PCA features

In [ ]:
arrays_by_set = {
    "compact": split_arrays(splits, compact, TARGET),
}
feature_names_by_set = {
    "compact": compact,
}

if len(expanded) > len(compact):
    arrays_by_set["expanded"] = split_arrays(splits, expanded, TARGET)
    feature_names_by_set["expanded"] = expanded

# Correlation-pruned compact feature set, fit on train only.
pruner = CorrelationPruner(threshold=0.95, feature_names=compact)
pruner.fit(splits["train"][compact])
corr_pruned_name = "compact_corr_pruned_0.95"

arrays_by_set[corr_pruned_name] = {}
for split_name, split in splits.items():
    X = pruner.transform(split[compact])
    y = split[TARGET].to_numpy(dtype=int)
    arrays_by_set[corr_pruned_name][split_name] = (X, y)

feature_names_by_set[corr_pruned_name] = list(pruner.get_feature_names_out())

print("features kept after correlation pruning:")
print(feature_names_by_set[corr_pruned_name])

In [ ]:
# PCA-reduced compact feature set, fit on train only.
n_pca = min(6, len(compact))
pca_reducer = build_dimred_pipeline(
    DimReductionConfig(kind="pca", n_components=n_pca, scale=True)
)
pca_reducer.fit(splits["train"][compact].to_numpy(dtype=float))

pca_name = f"compact_pca_{n_pca}"
arrays_by_set[pca_name] = {}
for split_name, split in splits.items():
    X = pca_reducer.transform(split[compact].to_numpy(dtype=float))
    y = split[TARGET].to_numpy(dtype=int)
    arrays_by_set[pca_name][split_name] = (X, y)

feature_names_by_set[pca_name] = [f"PC{i+1}" for i in range(n_pca)]

print(arrays_by_set.keys())

## 6. Run classical baseline suite

This skips dummy baselines. Models are intentionally simple/fast.

In [ ]:
configs = default_model_configs(random_state=42)

metrics, results = run_classical_suite(
    arrays_by_feature_set=arrays_by_set,
    feature_names_by_set=feature_names_by_set,
    configs=configs,
)

metrics.head(20)

In [ ]:
cols = [
    "feature_set",
    "model_name",
    "n_features",
    "val_pr_auc",
    "val_f1_class_1",
    "val_precision_class_1",
    "val_recall_class_1",
    "test_pr_auc",
    "test_f1_class_1",
    "test_precision_class_1",
    "test_recall_class_1",
]
metrics[cols].head(20)

## 7. Quick visual comparison

The primary ranking metric is validation PR-AUC because the stress class is imbalanced.

In [ ]:
plot_df = metrics.sort_values("val_pr_auc", ascending=True).tail(12)
labels = plot_df["feature_set"] + " :: " + plot_df["model_name"]

plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["val_pr_auc"])
plt.xlabel("Validation PR-AUC")
plt.title("Top classical baselines by validation PR-AUC")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = metrics.sort_values("test_pr_auc", ascending=True).tail(12)
labels = plot_df["feature_set"] + " :: " + plot_df["model_name"]

plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["test_pr_auc"])
plt.xlabel("Test PR-AUC")
plt.title("Top classical baselines by test PR-AUC")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Precision-recall and ROC curves for top validation models

Use these to see whether the models have ranking signal or just threshold artifacts.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

top_rows = metrics.head(6)
plt.figure(figsize=(7, 6))
ax = plt.gca()

for _, row in top_rows.iterrows():
    key = (row["feature_set"], row["model_name"])
    result = results[key]
    y_val = arrays_by_set[row["feature_set"]]["val"][1]
    PrecisionRecallDisplay.from_predictions(
        y_val,
        result.val_scores,
        name=f"{row['feature_set']}::{row['model_name']}",
        ax=ax,
    )

plt.title("Validation precision-recall curves")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
ax = plt.gca()

for _, row in top_rows.iterrows():
    key = (row["feature_set"], row["model_name"])
    result = results[key]
    y_val = arrays_by_set[row["feature_set"]]["val"][1]
    RocCurveDisplay.from_predictions(
        y_val,
        result.val_scores,
        name=f"{row['feature_set']}::{row['model_name']}",
        ax=ax,
    )

plt.title("Validation ROC curves")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Feature importance for best validation model

Permutation importance is computed on validation data. It is model-dependent and should be treated as diagnostic, not truth.

In [ ]:
best_row = metrics.iloc[0]
best_key = (best_row["feature_set"], best_row["model_name"])
best_result = results[best_key]
X_val, y_val = arrays_by_set[best_key[0]]["val"]

print("best by validation PR-AUC:", best_key)
print(best_row[cols])

In [ ]:
importance = permutation_importance_table(
    best_result,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
)

importance.head(20)

In [ ]:
plot_imp = importance.sort_values("importance_mean", ascending=True).tail(15)

plt.figure(figsize=(8, max(4, 0.35 * len(plot_imp))))
plt.barh(plot_imp["feature"], plot_imp["importance_mean"], xerr=plot_imp["importance_std"])
plt.xlabel("Permutation importance, validation AP")
plt.title(f"Permutation importance: {best_key[0]}::{best_key[1]}")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Save outputs and PDF report

This saves tables, figures, and a self-contained PDF report.

In [ ]:
REPORT_DIR = Path("reports/classical_baselines")
FIGURE_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"

ensure_dir(REPORT_DIR)
ensure_dir(FIGURE_DIR)
ensure_dir(TABLE_DIR)

dataset_summary = pd.DataFrame([{
    "n_rows": len(df),
    "n_columns": len(df.columns),
    "date_min": df["date"].min(),
    "date_max": df["date"].max(),
    "target": TARGET,
    "positive_rate": float(df[TARGET].mean()),
    "compact_features": len(compact),
    "expanded_features": len(expanded),
}])

dataset_summary.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)
metrics.to_csv(TABLE_DIR / "classical_metrics.csv", index=False)
corr_compact.to_csv(TABLE_DIR / "compact_feature_correlations.csv", index=False)
pca_summary.to_csv(TABLE_DIR / "pca_explained_variance.csv", index=False)
importance.to_csv(TABLE_DIR / "permutation_importance_best_model.csv", index=False)

print(TABLE_DIR)

In [ ]:
figure_paths = []

figure_paths.append(
    save_correlation_heatmap(
        df,
        compact,
        FIGURE_DIR / "compact_feature_correlation.png",
    )
)
figure_paths.append(
    save_metric_barplot(
        metrics,
        "val_pr_auc",
        FIGURE_DIR / "top_models_val_pr_auc.png",
    )
)
figure_paths.append(
    save_metric_barplot(
        metrics,
        "test_pr_auc",
        FIGURE_DIR / "top_models_test_pr_auc.png",
    )
)
figure_paths.append(
    save_pca_variance_plot(
        pca_summary,
        FIGURE_DIR / "pca_explained_variance.png",
    )
)
figure_paths.append(
    save_feature_importance_plot(
        importance,
        FIGURE_DIR / "permutation_importance_best_model.png",
        title=f"Permutation importance: {best_key[0]}::{best_key[1]}",
    )
)

# Saved PR/ROC figures from top validation models.
top_result_items = {}
for _, row in metrics.head(6).iterrows():
    key = (row["feature_set"], row["model_name"])
    top_result_items[f"{key[0]}::{key[1]}"] = results[key]

_, y_val_for_curves = arrays_by_set[metrics.iloc[0]["feature_set"]]["val"]
pr_path, roc_path = save_pr_roc_curves(
    top_result_items,
    y_val=y_val_for_curves,
    out_pr_path=FIGURE_DIR / "validation_pr_curves.png",
    out_roc_path=FIGURE_DIR / "validation_roc_curves.png",
)
figure_paths.extend([pr_path, roc_path])

figure_paths

In [ ]:
notes = [
    "This report benchmarks fast classical tabular models before further ESN/QRC work.",
    "Primary model-selection view is validation PR-AUC because the stress class is imbalanced.",
    "Test metrics are reported for diagnosis, not for choosing models post hoc.",
    "Feature redundancy is assessed through correlation, PCA explained variance, and permutation importance.",
]

pdf_path = build_pdf_report(
    output_pdf=REPORT_DIR / "classical_baseline_report.pdf",
    title="Classical Baselines and Feature Diagnostics",
    notes=notes,
    tables={
        "Dataset summary": dataset_summary,
        "Split summary": split_summary,
        "Top classical baselines": metrics[cols],
        "Highest compact feature correlations": corr_compact.head(15),
        "PCA explained variance": pca_summary,
        "Best-model permutation importance": importance.head(15),
    },
    figure_paths=figure_paths,
)

print(pdf_path)

## 11. Interpretation scratchpad

Use this cell to write conclusions after inspecting the outputs.

Suggested questions:

- Which simple model wins on validation PR-AUC?
- Does expanded feature set help or hurt?
- Does PCA preserve signal?
- Are feature importances concentrated in a few volatility/range/drawdown variables?
- Are reservoir models likely harmed by redundant inputs, or is the target itself the bottleneck?